# Module 05: The Beta Distribution on the Sphere

**Time:** ~25 minutes

In Module 04, we applied the Randomized Hadamard Transform (RHT) and saw that
the rotated coordinates look like a bell curve. But *which* bell curve exactly?
If we can pin down the exact distribution, we can compute the optimal Lloyd-Max
codebook **in advance** -- no training data needed.

**What this notebook covers:**
1. Why the distribution matters -- the link to data-oblivious quantization
2. Unit vectors on the hypersphere and the Beta distribution
3. What Beta(d/2, d/2) looks like for various d
4. The Gaussian approximation for large d
5. Empirical verification with RHT
6. Why this makes TurboQuant special
7. The dim < 64 edge case
8. Connection to the reference code

No exercise in this module -- the visualizations ARE the learning.

In [ ]:
%pip install -q torch numpy scipy plotly rich

In [ ]:
import numpy as np
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import beta as beta_dist
from scipy.stats import norm

import sys; sys.path.insert(0, "..")
from src.helpers import section_header

---
## 1. The Question

After applying the RHT, each coordinate of our rotated unit vector looks bell-shaped.
But "bell-shaped" isn't precise enough. Consider two very different scenarios:

- If the distribution is **known exactly**, we can precompute the optimal Lloyd-Max
  codebook once and reuse it forever. No training data, no calibration, no fitting.
- If the distribution is only **approximately known**, we'd need to estimate it from
  data -- which means a training step, which means overhead and potential mismatch.

TurboQuant gets the first scenario. The distribution is an exact mathematical fact,
not an empirical observation. Let's see why.

---
## 2. Unit Vectors and the Hypersphere

A **unit vector** lives on the surface of a d-dimensional sphere (the "hypersphere").
It satisfies $\|v\| = 1$, meaning all its coordinates $v_1, v_2, \ldots, v_d$ must
satisfy $v_1^2 + v_2^2 + \cdots + v_d^2 = 1$.

The RHT is an orthogonal transform -- it preserves norms and rotates the vector to
a "random-looking" orientation. After rotation, the vector is still on the sphere,
but it's as if we picked a **random point** on the sphere.

Here is the key mathematical fact (Lemma 1 in the TurboQuant paper):

> Each coordinate of a uniformly random point on the d-dimensional unit sphere
> follows a **Beta(d/2, d/2)** distribution, scaled to the interval [-1, 1].

More precisely: if $v$ is uniform on $S^{d-1}$, then each $v_i$ has the distribution
of $2B - 1$ where $B \sim \text{Beta}(d/2, d/2)$.

This is not an approximation. It's exact.

---
## 3. What Beta(d/2, d/2) Looks Like

The Beta(a, a) distribution on [0, 1] is symmetric around 1/2. When we shift it
to [-1, 1], it becomes symmetric around 0. The shape parameter `a = d/2` controls
how concentrated the distribution is:

- **a = 1** (d=2): Flat/uniform -- a circle in 2D
- **a = 2** (d=4): Slightly peaked
- **a = 5** (d=10): Clearly bell-shaped
- **a = 32** (d=64): Very tight around zero
- **a = 64** (d=128): Extremely concentrated

The higher the dimension, the more concentrated each coordinate is near zero.
This makes intuitive sense: with d coordinates that must sum-of-squares to 1,
each individual coordinate can't be too large.

In [ ]:
section_header("Beta(a, a) for various shape parameters",
               "Watch it go from flat to needle-sharp")

# Beta(a, a) on [0, 1], displayed on [-1, 1] via the transform x = 2*b - 1
x_01 = np.linspace(0.001, 0.999, 1000)
x_11 = 2 * x_01 - 1  # Map to [-1, 1]

a_values = [1, 2, 5, 16, 32, 64]
colors = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A", "#19D3F3"]

fig = go.Figure()

for a, color in zip(a_values, colors):
    rv = beta_dist(a, a)
    # PDF on [-1, 1]: f_{X}(x) = f_{B}((x+1)/2) * (1/2)
    pdf_vals = rv.pdf(x_01) / 2.0
    d_equiv = 2 * a
    fig.add_trace(go.Scatter(
        x=x_11, y=pdf_vals,
        mode="lines",
        name=f"a={a} (d={d_equiv})",
        line=dict(color=color, width=2),
    ))

fig.update_layout(
    title="Beta(a, a) on [-1, 1] -- each coordinate of a random unit vector",
    xaxis_title="Coordinate value",
    yaxis_title="Density",
    template="plotly_white",
    height=500,
    legend=dict(x=0.02, y=0.98),
)

fig.show()

Notice how the distribution starts flat at a=1 (the uniform case -- a 2D circle
has coordinates uniformly spread), then gets increasingly peaked at zero as `a` grows.
By a=64 (d=128), practically all the probability mass is squeezed into a tiny
region around zero. This is the **concentration of measure** phenomenon on the sphere.

---
## 4. The Gaussian Approximation

For large d, the Beta(d/2, d/2) distribution on [-1, 1] is very well approximated
by a Gaussian:

$$\text{Beta}(d/2, d/2) \text{ on } [-1, 1] \approx N\left(0, \frac{1}{d}\right)$$

Why? The variance of Beta(a, a) on [-1, 1] is $1/(2a + 1) = 1/(d + 1) \approx 1/d$
for large d. And by the central limit theorem on the sphere, the shape converges
to Gaussian.

This approximation is what allows the reference code to use precomputed N(0,1)
codebooks scaled by $1/\sqrt{d}$, rather than computing exact Beta codebooks
for every dimension.

In [ ]:
section_header("Beta vs Gaussian approximation",
               "Toggle traces to compare -- they converge as d grows")

dims = [64, 128, 256]
beta_colors = ["#636EFA", "#EF553B", "#00CC96"]
gauss_colors = ["#636EFA", "#EF553B", "#00CC96"]

fig = go.Figure()

for d, bc, gc in zip(dims, beta_colors, gauss_colors):
    a = d / 2.0
    rv = beta_dist(a, a)

    # Zoom in to where the mass is
    spread = 4.0 / np.sqrt(d)
    x_11 = np.linspace(-spread, spread, 1000)
    x_01 = (x_11 + 1) / 2.0

    # Beta PDF on [-1, 1]
    beta_pdf = rv.pdf(x_01) / 2.0

    # Gaussian N(0, 1/d)
    sigma = 1.0 / np.sqrt(d)
    gauss_pdf = norm.pdf(x_11, loc=0, scale=sigma)

    fig.add_trace(go.Scatter(
        x=x_11, y=beta_pdf,
        mode="lines",
        name=f"Beta({d//2},{d//2})  d={d}",
        line=dict(color=bc, width=2.5),
        legendgroup=f"d{d}",
    ))

    fig.add_trace(go.Scatter(
        x=x_11, y=gauss_pdf,
        mode="lines",
        name=f"N(0, 1/{d})  d={d}",
        line=dict(color=gc, width=2, dash="dash"),
        legendgroup=f"d{d}",
    ))

fig.update_layout(
    title="Beta(d/2, d/2) vs N(0, 1/d) -- nearly indistinguishable for d >= 64",
    xaxis_title="Coordinate value",
    yaxis_title="Density",
    template="plotly_white",
    height=500,
    legend=dict(x=0.02, y=0.98),
)

fig.show()

Click the legend entries to toggle individual traces. For each dimension, the solid
line (exact Beta) and dashed line (Gaussian approximation) are nearly on top of
each other. By d=256 they are effectively identical.

This is why the reference code uses $N(0, 1)$ codebooks scaled by $1/\sqrt{d}$
for dimensions 64 and above -- the Gaussian approximation is excellent.

---
## 5. Empirical Verification

Theory is nice, but let's see it with our own eyes. We will:
1. Generate 5000 random unit vectors in d=128
2. Apply the RHT
3. Plot a histogram of one coordinate across all 5000 vectors
4. Overlay the theoretical Beta and Gaussian PDFs

If the theory is right, the histogram should match both curves beautifully.

In [ ]:
import math

section_header("Empirical verification: RHT coordinates",
               "5000 random unit vectors in d=128, after RHT")

d = 128
n_vectors = 5000

# Step 1: Generate random unit vectors
torch.manual_seed(42)
vectors = torch.randn(n_vectors, d)
vectors = vectors / vectors.norm(dim=-1, keepdim=True)  # Normalize to unit length

# Step 2: Apply RHT (sign flip + FWHT)
# We'll implement the essentials inline so this notebook is self-contained.
def fwht(x: torch.Tensor) -> torch.Tensor:
    """Fast Walsh-Hadamard Transform, normalized."""
    d = x.shape[-1]
    result = x.clone()
    h = 1
    while h < d:
        result = result.view(*result.shape[:-1], -1, 2 * h)
        left = result[..., :h]
        right = result[..., h:]
        result = torch.cat([left + right, left - right], dim=-1)
        result = result.view(*x.shape)
        h *= 2
    return result / math.sqrt(d)

# Random sign flips (Rademacher)
gen = torch.Generator()
gen.manual_seed(0)
signs = (torch.randint(0, 2, (d,), generator=gen) * 2 - 1).float()

# Apply: sign flip, then FWHT
rotated = fwht(vectors * signs)

print(f"Input shape:   {vectors.shape}")
print(f"Rotated shape: {rotated.shape}")
print(f"Norms preserved: {torch.allclose(vectors.norm(dim=-1), rotated.norm(dim=-1), atol=1e-5)}")

In [ ]:
section_header("Histogram vs theory",
               "Coordinate 0 of 5000 rotated unit vectors, d=128")

# Extract one coordinate from all rotated vectors
coord_idx = 0
samples = rotated[:, coord_idx].numpy()

# Theoretical PDFs
a = d / 2.0  # = 64
rv = beta_dist(a, a)
sigma = 1.0 / np.sqrt(d)

spread = 4.0 * sigma
x_range = np.linspace(-spread, spread, 500)
x_01 = (x_range + 1) / 2.0

beta_pdf = rv.pdf(x_01) / 2.0
gauss_pdf = norm.pdf(x_range, loc=0, scale=sigma)

fig = go.Figure()

# Histogram of empirical samples
fig.add_trace(go.Histogram(
    x=samples,
    nbinsx=60,
    histnorm="probability density",
    name="Empirical (5000 vectors)",
    marker_color="rgba(99, 110, 250, 0.5)",
    marker_line=dict(color="rgba(99, 110, 250, 0.8)", width=1),
))

# Beta PDF
fig.add_trace(go.Scatter(
    x=x_range, y=beta_pdf,
    mode="lines",
    name=f"Beta({int(a)}, {int(a)}) on [-1, 1]",
    line=dict(color="#EF553B", width=3),
))

# Gaussian PDF
fig.add_trace(go.Scatter(
    x=x_range, y=gauss_pdf,
    mode="lines",
    name=f"N(0, 1/{d})",
    line=dict(color="#00CC96", width=3, dash="dash"),
))

fig.update_layout(
    title=f"Coordinate {coord_idx} of {n_vectors} rotated unit vectors (d={d})",
    xaxis_title="Coordinate value",
    yaxis_title="Density",
    template="plotly_white",
    height=500,
    bargap=0.05,
    legend=dict(x=0.55, y=0.98),
)

fig.show()

The histogram of empirical RHT coordinates matches both the exact Beta(64, 64) PDF
(solid red) and the Gaussian approximation N(0, 1/128) (dashed green) extremely well.
The two theoretical curves are so close they overlap almost perfectly.

This isn't a coincidence or a lucky draw. It's a mathematical certainty: the RHT
makes each coordinate of a unit vector follow Beta(d/2, d/2), which for d=128
is nearly indistinguishable from N(0, 1/128).

---
## 6. Why This Matters for TurboQuant

This is the punchline of the entire module, and one of the key insights of
the TurboQuant paper.

Since we **know** the distribution of each coordinate before seeing any data:

1. **Precompute the codebook.** We can run Lloyd-Max on the known Beta(d/2, d/2)
   distribution (or its Gaussian approximation) to get the MSE-optimal quantizer.
   This is a one-time cost.

2. **Data-oblivious.** The codebook depends only on `(bits, dimension)`, NOT on the
   actual vectors being quantized. Any unit vector in d dimensions will have
   the same coordinate distribution after RHT.

3. **No training step.** Unlike product quantization (PQ) or learned quantization
   methods, TurboQuant needs no training set, no calibration pass, no fitting.
   You just need to know d and the desired bit width.

4. **No distribution shift.** Since the codebook is derived from a mathematical fact
   (not a data sample), it never goes stale. New data from a different domain
   works just as well.

This is fundamentally different from methods like product quantization, which
require:
- A representative training set
- A k-means fitting step (often expensive)
- Re-training when data distribution changes

---
## 7. The dim < 64 Case

The Gaussian approximation works wonderfully for large dimensions, but what
about small ones? Let's see what happens at d=8 (a=4), where the approximation
is noticeably off.

In [ ]:
section_header("Small dimension: Beta vs Gaussian at d=8",
               "The Gaussian approximation breaks down for small d")

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["d=8: Noticeable mismatch", "d=128: Nearly perfect"],
    horizontal_spacing=0.1,
)

for col, d_val in enumerate([8, 128], start=1):
    a = d_val / 2.0
    rv = beta_dist(a, a)
    sigma = 1.0 / np.sqrt(d_val)

    spread = 4.0 * sigma
    x_range = np.linspace(-spread, spread, 500)
    x_01 = (x_range + 1) / 2.0

    beta_pdf = rv.pdf(x_01) / 2.0
    gauss_pdf = norm.pdf(x_range, loc=0, scale=sigma)

    fig.add_trace(go.Scatter(
        x=x_range, y=beta_pdf,
        mode="lines",
        name=f"Beta({int(a)},{int(a)})",
        line=dict(color="#EF553B", width=2.5),
        legendgroup=f"beta{d_val}",
        showlegend=(col == 1),
    ), row=1, col=col)

    fig.add_trace(go.Scatter(
        x=x_range, y=gauss_pdf,
        mode="lines",
        name=f"N(0, 1/{d_val})",
        line=dict(color="#00CC96", width=2.5, dash="dash"),
        legendgroup=f"gauss{d_val}",
        showlegend=(col == 1),
    ), row=1, col=col)

fig.update_layout(
    title="Beta vs Gaussian: small d vs large d",
    template="plotly_white",
    height=400,
    legend=dict(x=0.02, y=0.98),
)
fig.update_xaxes(title_text="Coordinate value", row=1, col=1)
fig.update_xaxes(title_text="Coordinate value", row=1, col=2)
fig.update_yaxes(title_text="Density", row=1, col=1)

fig.show()

At d=8, the Beta(4, 4) distribution has heavier tails and a flatter top than
N(0, 1/8). Using the Gaussian codebook here would introduce unnecessary
quantization error.

The reference code handles this by computing **exact Beta codebooks** for d < 64
and only using the Gaussian shortcut for d >= 64. This gives optimal quantization
at every dimension.

---
## 8. Connection to the Reference Code

Here is the `get_codebook` function from the TurboQuant reference implementation
(`turboquant/codebook.py`). Notice how the dim >= 64 branching logic directly
implements the Gaussian approximation we just explored:

In [ ]:
section_header("Reference code: get_codebook()",
               "The dim >= 64 branch uses the Gaussian approximation")

# This is the actual logic from turboquant/codebook.py, simplified for clarity.
# _NORMAL_CODEBOOKS contains precomputed Lloyd-Max codebooks for N(0,1).
# _lloyd_max_beta() computes exact codebooks for Beta(d/2, d/2).

code = '''
def get_codebook(bits: int, dim: int) -> Codebook:
    """Get the Lloyd-Max codebook for a given bit width and dimension.

    For dim >= 64, uses the Gaussian approximation (precomputed N(0,1)
    codebook scaled by 1/sqrt(d)). For dim < 64, computes exact
    codebook for Beta(d/2, d/2).
    """
    if dim >= 64:
        # Gaussian approximation: N(0, 1/d) ~ Beta(d/2, d/2) on [-1, 1]
        # Reuse precomputed N(0,1) codebook, just scale by 1/sqrt(d)
        cb = _NORMAL_CODEBOOKS[bits]
        scale = 1.0 / np.sqrt(dim)
        return Codebook(
            boundaries=cb.boundaries * scale,
            centroids=cb.centroids * scale,
        )
    else:
        # Small dimension: Gaussian approximation isn't good enough.
        # Compute exact Lloyd-Max codebook for Beta(d/2, d/2).
        return _lloyd_max_beta(bits, dim)
'''

print(code)

The key points:

- For **dim >= 64**: A single precomputed N(0,1) codebook is scaled by $1/\sqrt{d}$.
  This is fast (just a multiplication) and accurate (the Gaussian approximation
  is excellent at these dimensions).

- For **dim < 64**: An exact Lloyd-Max codebook is computed for Beta(d/2, d/2).
  This is slower (requires numerical integration) but necessary because the
  Gaussian approximation is too loose.

- The codebook depends only on **(bits, dim)** -- not on any training data.
  This is what makes TurboQuant data-oblivious.

---
## Interactive Explorer: Beta -> Gaussian Convergence

Use the slider below to see how Beta(d/2, d/2) converges to the Gaussian
as dimension increases. Each frame shows the exact Beta PDF (solid) and
the Gaussian approximation (dashed).

In [ ]:
section_header("Interactive: Beta -> Gaussian convergence",
               "Drag the slider to change dimension")

dim_values = [4, 8, 16, 32, 64, 128, 256, 512]

fig = go.Figure()

# Create traces for each dimension (only first visible)
for i, d_val in enumerate(dim_values):
    a = d_val / 2.0
    rv = beta_dist(a, a)
    sigma = 1.0 / np.sqrt(d_val)

    spread = max(4.0 * sigma, 0.3)  # Ensure we see something for small d
    x_range = np.linspace(-spread, spread, 500)
    x_01 = np.clip((x_range + 1) / 2.0, 1e-10, 1 - 1e-10)

    beta_pdf = rv.pdf(x_01) / 2.0
    gauss_pdf = norm.pdf(x_range, loc=0, scale=sigma)

    visible = (i == 0)

    fig.add_trace(go.Scatter(
        x=x_range, y=beta_pdf,
        mode="lines",
        name=f"Beta({int(a)},{int(a)})",
        line=dict(color="#EF553B", width=3),
        visible=visible,
    ))

    fig.add_trace(go.Scatter(
        x=x_range, y=gauss_pdf,
        mode="lines",
        name=f"N(0, 1/{d_val})",
        line=dict(color="#636EFA", width=3, dash="dash"),
        visible=visible,
    ))

# Create slider steps
steps = []
for i, d_val in enumerate(dim_values):
    a = d_val / 2.0
    step = dict(
        method="update",
        args=[
            {"visible": [False] * (2 * len(dim_values))},
            {"title": f"d={d_val}: Beta({int(a)},{int(a)}) vs N(0, 1/{d_val})"},
        ],
        label=str(d_val),
    )
    step["args"][0]["visible"][2 * i] = True      # Beta trace
    step["args"][0]["visible"][2 * i + 1] = True   # Gaussian trace
    steps.append(step)

fig.update_layout(
    sliders=[dict(
        active=0,
        currentvalue=dict(prefix="Dimension d = "),
        pad=dict(t=60),
        steps=steps,
    )],
    title=f"d={dim_values[0]}: Beta({dim_values[0]//2},{dim_values[0]//2}) vs N(0, 1/{dim_values[0]})",
    xaxis_title="Coordinate value",
    yaxis_title="Density",
    template="plotly_white",
    height=550,
    showlegend=True,
)

fig.show()

At d=4, the Beta and Gaussian are visibly different -- the Beta has a flatter top
and is bounded at [-1, 1] while the Gaussian extends beyond. By d=64, you have
to look very carefully to see any difference. By d=256+, they are indistinguishable.

This visual tells you exactly WHY the reference code uses d=64 as the cutoff.

---
## Recap

In this module we established the theoretical foundation for TurboQuant's
data-oblivious quantization:

1. **After RHT, each coordinate of a unit vector follows Beta(d/2, d/2) on [-1, 1].**
   This is an exact mathematical fact, not an approximation.

2. **For large d (>= 64), Beta(d/2, d/2) is approximately N(0, 1/d).**
   This allows using a single precomputed N(0,1) codebook, scaled by 1/sqrt(d).

3. **For small d (< 64), the Gaussian approximation is inaccurate.**
   The reference code computes exact Beta codebooks in this regime.

4. **Knowing the distribution in advance eliminates the need for training data.**
   This is what makes TurboQuant fundamentally different from methods that require
   calibration sets.

**Next up:** Module 06 -- Lloyd-Max Quantization, where we use this distribution
knowledge to build the optimal scalar quantizer.